# Experimento 6 — Remoção das Variáveis Categóricas com Balanceamento

Após os resultados obtidos no experimento anterior, observou-se que mesmo utilizando apenas:

* temperatura;
* ortofosfato;
* nitrogênio;

o modelo ainda conseguiu manter desempenho relativamente alto.

No entanto, também ficou evidente que a ausência das variáveis categóricas fez o Random Forest favorecer ainda mais a classe `Excelente`, consequência direta do forte desbalanceamento presente no dataset.

Além disso, as classes minoritárias, principalmente `Crítica`, continuaram apresentando grande dificuldade de identificação.

Diante disso, surgiu a necessidade de investigar como o modelo se comporta quando:

* utiliza apenas variáveis numéricas;
* recebe balanceamento durante o treinamento.

Neste experimento, foram mantidas apenas as variáveis:

* temperatura;
* ortofosfato;
* nitrogênio.

Ao mesmo tempo, foi aplicado balanceamento para tentar reduzir a influência excessiva da classe majoritária durante o processo de aprendizado.

O principal objetivo deste experimento é analisar:

* se o balanceamento melhora a identificação das classes minoritárias;
* se o recall da classe `Crítica` aumenta;
* como o modelo reage sem informações contextuais;
* até que ponto apenas variáveis físico-químicas conseguem sustentar o processo de classificação;
* se o balanceamento consegue reduzir a tendência do modelo de prever excessivamente a classe `Excelente`.

Este experimento também é importante para entender melhor o papel das variáveis categóricas no processo de generalização do modelo.

Nos experimentos anteriores, informações como:

* país;
* tipo de corpo hídrico;

ajudavam o algoritmo a contextualizar determinados padrões ambientais.

Agora, sem essas informações, o modelo precisa depender exclusivamente das relações numéricas existentes entre:

* temperatura;
* ortofosfato;
* nitrogênio;
* qualidade da água.

Dessa forma, este experimento permite investigar simultaneamente:

* impacto do balanceamento;
* impacto da remoção do contexto ambiental;
* capacidade preditiva das variáveis físico-químicas;
* comportamento do Random Forest diante de classes altamente desbalanceadas.


## Preparação do ambiente


In [1]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
# preparando o ambiente para machine learning
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
# definição de x e y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Nitrogen (mg/l)"
    ]
]

y = df["conama_status"]

In [5]:
#train_test split

SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


Treino: (113119, 3)
Teste: (28280, 3)


In [6]:
model = Pipeline(
    steps=[

       (
            "classifier",
            RandomForestClassifier(
                random_state=SEED,
                class_weight="balanced"
            )
        )
    ]
)

In [7]:
model.fit(X_train, y_train)

Pipeline(steps=[('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi realizada a avaliação das métricas no conjunto de treino.

Essa etapa é importante porque permite comparar o comportamento do modelo entre treino e teste, ajudando na identificação de possíveis sinais de overfitting.

Por esse motivo, não basta apenas observar uma alta acurácia no treino. O ideal é que os resultados de treino e teste permaneçam relativamente próximos, indicando que o modelo conseguiu aprender padrões mais generalizáveis ao invés de apenas memorizar os dados.

Além da acurácia, também foram analisadas métricas como precision, recall e F1-score, permitindo observar como o modelo se comporta em cada classe individualmente.

In [8]:
y_train_pred = model.predict(X_train)

In [9]:
train_accuracy = accuracy_score(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

Train Accuracy:
0.8788974442843377


In [10]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.8788974442843377
Train Precision:
0.916812418659294
Train Recall:
0.8788974442843377
Train F1:
0.8902453854803087
Train Confusion Matrix:
[[ 7207   229    29    95]
 [ 2890 17394    62  1332]
 [  119     4   978     5]
 [ 5664  3040   230 73841]]


In [11]:
y_pred = model.predict(X_test)

In [12]:
# Teste com 5 variáveis - com balanceamento
print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7102545968882602

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.23      0.44      0.31      1890
         Boa       0.46      0.37      0.41      5419
     Crítica       0.07      0.04      0.05       277
   Excelente       0.85      0.83      0.84     20694

    accuracy                           0.71     28280
   macro avg       0.40      0.42      0.40     28280
weighted avg       0.73      0.71      0.72     28280


Confusion Matrix:
[[  832   518    22   518]
 [  953  1994    48  2424]
 [   66   118    11    82]
 [ 1702  1665    78 17249]]


## Resultados Obtidos — Experimento 6

Os resultados do sexto experimento mostraram mudanças importantes no comportamento do modelo após a combinação entre:

* remoção das variáveis categóricas;
* utilização apenas de variáveis físico-químicas;
* aplicação de balanceamento durante o treinamento.

A acurácia de treino ficou em aproximadamente 88%, enquanto a acurácia de teste ficou em torno de 71%.

Em comparação ao experimento anterior, percebe-se uma redução tanto na acurácia de treino quanto na acurácia de teste.

No entanto, essa redução não necessariamente representa piora do modelo.

Na prática, isso indica que o Random Forest deixou de favorecer excessivamente a classe `Excelente` e passou a distribuir melhor sua atenção entre as demais classes.

Um dos pontos mais importantes observados neste experimento foi o aumento significativo do recall da classe `Atenção`.

No experimento anterior, grande parte das amostras intermediárias acabava sendo empurrada para a classe `Excelente`.

Com o balanceamento, o modelo passou a identificar muito mais exemplos da classe `Atenção`, mostrando comportamento mais equilibrado.

Isso demonstra que o balanceamento ajudou o algoritmo a prestar mais atenção em padrões ambientais menos dominantes no dataset.

Ao mesmo tempo, o recall da classe `Excelente` diminuiu em comparação ao experimento anterior.

Isso acontece porque o modelo passou a “arriscar mais” classificações em classes minoritárias, reduzindo a tendência conservadora de prever quase tudo como `Excelente`.

Esse comportamento é esperado em cenários de balanceamento.

A classe `Boa` também apresentou comportamento relativamente mais equilibrado, conseguindo manter métricas razoáveis mesmo sem as variáveis categóricas.

Já a classe `Crítica` continuou sendo a mais difícil de identificar.

Mesmo com balanceamento, a quantidade extremamente pequena de exemplos críticos ainda limita bastante a capacidade de aprendizado do modelo.

Outro ponto importante observado neste experimento foi a influência das variáveis categóricas nos resultados anteriores.

Ao remover:

* `Country`;
* `Waterbody Type`;

o modelo perdeu parte do contexto ambiental que ajudava na identificação de padrões específicos.

Isso sugere que variáveis contextuais possuem papel importante no processo de classificação da qualidade da água.

Mesmo assim, as variáveis:

* temperatura;
* ortofosfato;
* nitrogênio;

continuaram apresentando capacidade preditiva relevante, mostrando que parâmetros físico-químicos ainda conseguem carregar informações ambientais importantes.

De forma geral, este experimento mostrou que:

* o balanceamento reduziu parcialmente o favorecimento excessivo da classe `Excelente`;
* o modelo passou a identificar melhor classes intermediárias;
* a remoção das variáveis categóricas reduziu parte da contextualização ambiental;
* as variáveis físico-químicas continuam possuindo forte capacidade preditiva;
* o dataset desbalanceado continua sendo um dos principais desafios para identificação da classe `Crítica`.
